<a href="https://colab.research.google.com/github/Naylet92/Estudio_ambiental/blob/main/Quila/QLA_L_2000_ver3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DESCRIPCIÓN DEL PROCESAMIENTO DE IMÁGENES LANDSAT 5(AÑO 2000)

##1. Información general de las imágenes
Para el análisis multitemporal se utilizaron imágenes de la colección Landsat 5 TM Collection 2 Level-2 (reflectancia superficial) correspondientes al año 2000. Se definieron dos temporadas de estudio:

Temporada seca: 1 de marzo al 31 de mayo-----> Evaluar condiciones de estrés hídrico y vegetación en estiaje

Temporada húmeda: 1 de agosto al 30 de noviembre-----> 	Evaluar el máximo desarrollo vegetal post-lluvias

El área de estudio corresponde a la Sierra de Quila, delimitada por un polígono vectorial cargado desde un archivo GeoJSON. Todas las imágenes fueron recortadas a esta zona de interés.

##2. Procesamiento aplicado
El flujo de procesamiento incluyó las siguientes etapas:

2.1. Enmascaramiento de nubes y sombras
Se utilizó la banda de calidad QA_PIXEL para excluir píxeles afectados por nubes y sombras, asegurando que solo información válida contribuyera al mosaico final.

2.2. Corrección topográfica
Se aplicó una corrección por iluminación topográfica utilizando el modelo digital de elevación SRTM (30 m). El método corrige las diferencias de iluminación causadas por la pendiente y orientación del terreno, normalizando las reflectancias con el ángulo solar y el coseno del ángulo de incidencia.

2.3. Composición de mediana
Para cada temporada se generó un mosaico de mediana utilizando todas las imágenes disponibles. Este enfoque reduce el ruido y minimiza la influencia de artefactos residuales de nubes o sombras.

2.4. Cálculo de índices y componentes
A partir del mosaico de mediana se derivaron:

NDVI (Índice de Vegetación de Diferencia Normalizada)
NDVI = (NIR - Rojo) / (NIR + Rojo)

Componentes de Tasseled Cap (Brightness, Greenness, Wetness)
Utilizando los coeficientes estándar para Landsat 5 TM (Crist, 1985), que permiten separar información estructural, de verdor y de humedad.
| Componente | B1 (Azul) | B2 (Verde) | B3 (Rojo) | B4 (NIR) | B5 (SWIR1) | B7 (SWIR2) |
|------------|-----------|------------|-----------|----------|------------|------------|
| **Brightness** | 0.3037 | 0.2793 | 0.4743 | 0.5585 | 0.5082 | 0.1863 |
| **Greenness** | -0.2848 | -0.2435 | -0.5436 | 0.7243 | 0.0840 | -0.1800 |
| **Wetness** | 0.1509 | 0.1973 | 0.3279 | 0.3406 | -0.7112 | -0.4572 |

##3. Imágenes combinadas por temporada
Temporada seca (8 imágenes)
Se combinaron las siguientes escenas del año 2000:

| 🌦️ Temporada | 📅 Rango de fechas | 📷 Imágenes | 🗓️ Fechas de adquisición |
|-------------|-------------------|-------------|-------------------------|
| **Seca** | 01/03 – 31/05 | 8 | `2000-04-03` · `2000-04-10` · `2000-04-19` · `2000-04-26` · `2000-05-05` · `2000-05-12` · `2000-05-21` · `2000-05-28` |
| **Húmeda** | 01/08 – 30/11 | 11 | `2000-08-09` · `2000-08-16` · `2000-08-25` · `2000-09-01` · `2000-09-10` · `2000-09-17` · `2000-09-26` · `2000-10-03` · `2000-10-12` · `2000-10-19` · `2000-11-29` |



##4. Parámetros de visualización obtenidos
Los valores estadísticos de las bandas resultantes fueron:
| Variable | Temporada | Mínimo | Máximo | Paleta | Interpretación del rango |
|----------|-----------|--------|--------|--------|--------------------------|
| **NDVI** | Seca | 0.10 | 0.79 | Rojo → Amarillo → Verde | Desde suelo desnudo hasta vegetación densa |
| | Húmeda | 0.12 | 0.87 | Rojo → Amarillo → Verde | Mayor vigor vegetal en el extremo superior |
| **TC Brightness** | Seca | 0.00 | 1.40 | Blanco → Café | Contraste entre sombras y superficies brillantes |
| | Húmeda | 0.00 | 1.39 | Blanco → Café | Rango similar a temporada seca |
| **TC Greenness** | Seca | -0.03 | 0.33 | Blanco → Verde oscuro | Vegetación presente pero limitada |
| | Húmeda | -0.01 | 0.84 | Blanco → Verde oscuro | **Mucha más vegetación activa** |
| **TC Wetness** | Seca | -0.50 | 0.03 | Café → Blanco → Azul | Condiciones muy secas (valores negativos) |
| | Húmeda | -0.27 | 0.08 | Café → Blanco → Azul | Mayor humedad superficial |

##5. Comparación de resultados
| Sensor | Temporada | NDVI | TC Brightness | TC Greenness | TC Wetness | Evaluación general |
|--------|-----------|------|---------------|--------------|------------|-------------------|
| **L5** | Seca | 0.10 - 0.79 | 0.00 - 1.40 | -0.03 - 0.33 | -0.50 - 0.03 | ✅ Correcto, rangos realistas |
| | Húmeda | 0.12 - 0.87 | 0.00 - 1.39 | -0.01 - 0.84 | -0.27 - 0.08 | ✅ Excelente, respuesta fenológica clara |

## Selección y procesamiento de imágenes Landsat (año 2000)


##Definir Variables

In [1]:
PREFIJO     = 'QLA'
NOMBRE      = 'Sierra_Quila'
#RUTA_AOI    = '/content/drive/MyDrive/ANP/AOI/Quila'
RUTA_AOI    = '/content/drive/MyDrive/Colab Data/Naylet/AOI'

RUTA_GEOJSON = f'{RUTA_AOI}/aoi_{PREFIJO}.geojson'
RUTA_BBOX   = f'{RUTA_AOI}/{PREFIJO}_bbox.geojson'
RUTA_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas.csv'

# CRS geográfico y UTM
CRS_GEO     = 4326
CRS_UTM     = 32613

# Proyecto de Google Earth Engine
#GEE_PROJECT = 'ee-nayleths'
GEE_PROJECT = 'ee-vshalisko'

AÑO           = 2000
COLECCION_L5     = 'LANDSAT/LT05/C02/T1_L2'   # Landsat 5 TM para el año 2000
COLECCION_L7    = 'LANDSAT/LE07/C02/T1_L2'    # Landsat 7 ETM para el año 2000

# Temporadas de análisis
TEMPORADA_SECA   = ('03-01', '05-31')
TEMPORADA_HUMEDA = ('08-01', '11-30')

# Bandas espectrales de reflectancia superficial
BANDAS_SR     = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
BANDAS_CORR   = [b + '_corr' for b in BANDAS_SR]

# Coeficientes Tasseled Cap (Landsat 5 TM)
TC_BRIGHT_L5 = [0.3037, 0.2793, 0.4743, 0.5585, 0.5082, 0.1863]
TC_GREEN_L5  = [-0.2848, -0.2435, -0.5436, 0.7243, 0.0840, -0.1800]
TC_WET_L5    = [0.1509, 0.1973, 0.3279, 0.3406, -0.7112, -0.4572]

# Coeficientes Tasseled Cap (Landsat 7 ETM+)
TC_BRIGHT_L7 = [0.3561, 0.3972, 0.3904, 0.6966, 0.2286, 0.1596]
TC_GREEN_L7  = [-0.3344, -0.3544, -0.4556, 0.6966, -0.0242, -0.2630]
TC_WET_L7    = [0.2626, 0.2141, 0.0926, 0.0656, -0.7629, -0.5388]

# Escala de exportación (m)
ESCALA        = 30

# Zoom inicial del mapa
ZOOM        = 12

# Estilo del área de estudio
STYLE_AREA  = {'color': 'blue', 'fillColor': '#0000ff30', 'weight': 1.5}

# Estilo del rectángulo rojo o bounding box
STYLE_BBOX  = {'color': 'red', 'fillColor': '#00000000', 'weight': 2.5}

# Título del mapa
MAP_TITLE   = 'Área de estudio de Sierra de Quila'

#RUTA_IMAGENES = '/content/drive/MyDrive/ANP/Landsat'
RUTA_IMAGENES = 'Colab Data NDC 2020'

In [2]:
import ee
import math
import json
import geemap
import geopandas as gpd
from shapely.geometry import box
from google.colab import drive


# Montar Google Drive
drive.mount('/content/drive')

# Autenticar e inicializar GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("✓ Entorno listo")

Mounted at /content/drive
✓ Entorno listo


## 4. Funciones de preprocesamiento

In [3]:
# Enmascarar nubes y sombras
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_mask        = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud_shadow_mask.And(cloud_mask))


def correccion_topografica(image, region):
    dem        = ee.Image('USGS/SRTMGL1_003').clip(region)
    terreno    = ee.Terrain.products(dem)

    pendiente   = terreno.select('slope').multiply(math.pi / 180)
    orientacion = terreno.select('aspect').multiply(math.pi / 180)

    az_solar   = ee.Number(image.get('SUN_AZIMUTH')).multiply(math.pi / 180)
    elev_solar = ee.Number(image.get('SUN_ELEVATION')).multiply(math.pi / 180)

    cos_i = (
        pendiente.cos().multiply(elev_solar.sin())
        .add(
            pendiente.sin()
            .multiply(elev_solar.cos())
            .multiply(orientacion.subtract(az_solar).cos())
        )
    )
    cos_i_safe = cos_i.max(0.2)

    # Factor de escala para Collection 2 (DN * 0.0000275 - 0.2)
    im_bandas = image.select(BANDAS_SR).multiply(0.0000275).add(-0.2)

    bandas_corr = (
    ee.Image.cat([
        im_bandas.select(b).divide(cos_i_safe).rename(BANDAS_CORR[i])
        for i, b in enumerate(BANDAS_SR)
        ])
        .set('SUN_AZIMUTH',   image.get('SUN_AZIMUTH'))
        .set('SUN_ELEVATION', image.get('SUN_ELEVATION'))
        .cast({b: 'double' for b in BANDAS_CORR})
    )

    return bandas_corr

def tasseled_cap(image_corr, coefs, nombre):
    partes = ee.Image.cat([
        image_corr.select(b).multiply(c)
        for b, c in zip(BANDAS_CORR, coefs)
    ])
    return partes.reduce(ee.Reducer.sum()).rename(nombre)

### Procesamiento por temporada

In [4]:
gdf = gpd.read_file(RUTA_BBOX)
gcojson_str = gdf.to_crs(epsg=CRS_GEO).to_json()
region = ee.FeatureCollection(json.loads(gcojson_str)).geometry()

#Procesar temporada Seca y de Húmedad
def procesar_temporada(nombre_temporada, fecha_ini, fecha_fin):
    print(f"\n── Temporada: {nombre_temporada} ({fecha_ini} → {fecha_fin}) ──")

    fecha_inicio = f'{AÑO}-{fecha_ini}'
    fecha_fin_   = f'{AÑO}-{fecha_fin}'

    col_l5 = (ee.ImageCollection(COLECCION_L5)
             .filterDate(fecha_inicio, fecha_fin_)
             .filterBounds(region))

    col_l7 = (ee.ImageCollection(COLECCION_L7)
             .filterDate(fecha_inicio, fecha_fin_)
             .filterBounds(region))

    n_l5 = col_l5.size().getInfo()
    n_l7 = col_l7.size().getInfo()

    print(f"  Imágenes Landsat 5: {n_l5}")
    if n_l5 > 0:
        fechas_l5 = col_l5.aggregate_array('DATE_ACQUIRED').getInfo()
        info_l5 = col_l5.aggregate_array('LANDSAT_PRODUCT_ID').getInfo()
        for f in sorted(fechas_l5):
            print(f"    · L5 {f}")
        for i in info_l5:
            print(i)
    else:
        print("    Sin imágenes L5")

    print(f"  Imágenes Landsat 7: {n_l7}")
    if n_l7 > 0:
        fechas_l7 = col_l7.aggregate_array('DATE_ACQUIRED').getInfo()
        info_l7 = col_l7.aggregate_array('LANDSAT_PRODUCT_ID').getInfo()
        for f in sorted(fechas_l7):
            print(f"    · L7 {f}")
        for i in info_l7:
            print(i)
    else:
        print("    Sin imágenes L7")


    #  Procesar L5 si hay imágenes
    img_l5 = None
    mosaico_l5 = None
    if n_l5 > 0:
        col_l5_limpia = col_l5.map(mask_clouds).map(lambda img: correccion_topografica(img, region))
        col_l5_corr = col_l5_limpia.select(BANDAS_CORR)
        mosaico_l5 = col_l5_corr.median().clip(region)
        img_l5 = procesar_bandas_l5(mosaico_l5)
        print(f"  L5 procesado correctamente")
    else:
        print(f"  L5: Sin datos para esta temporada")

    #  Procesar L7 si hay imágenes
    img_l7 = None
    mosaico_l7 = None
    if n_l7 > 0:
        col_l7_limpia = col_l7.map(mask_clouds).map(lambda img: correccion_topografica(img, region))
        col_l7_corr = col_l7_limpia.select(BANDAS_CORR)
        mosaico_l7 = col_l7_corr.median().clip(region)
        img_l7 = procesar_bandas_l7(mosaico_l7)
        print(f"  L7 procesado correctamente")
    else:
        print(f"  L7: Sin datos para esta temporada")

    return img_l5, img_l7, mosaico_l5, mosaico_l7

def procesar_bandas_l5(mosaico):
    """Función auxiliar para calcular NDVI y Tasseled Cap"""
    # ── NDVI
    nir  = mosaico.select('SR_B4_corr')
    red  = mosaico.select('SR_B3_corr')
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    # ── Tasseled Cap
    tc_b = tasseled_cap(mosaico, TC_BRIGHT_L5, 'TC_Brightness')
    tc_g = tasseled_cap(mosaico, TC_GREEN_L5,  'TC_Greenness')
    tc_w = tasseled_cap(mosaico, TC_WET_L5,    'TC_Wetness')

    resultado = (ee.Image.cat([ndvi, tc_b, tc_g, tc_w])
                   .clip(region)
                   .cast({'NDVI': 'double', 'TC_Brightness': 'double',
                          'TC_Greenness': 'double', 'TC_Wetness': 'double'}))
    return resultado

def procesar_bandas_l7(mosaico):
    """Función auxiliar para calcular NDVI y Tasseled Cap"""
    # ── NDVI
    nir  = mosaico.select('SR_B4_corr')
    red  = mosaico.select('SR_B3_corr')
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    # ── Tasseled Cap
    tc_b = tasseled_cap(mosaico, TC_BRIGHT_L7, 'TC_Brightness')
    tc_g = tasseled_cap(mosaico, TC_GREEN_L7,  'TC_Greenness')
    tc_w = tasseled_cap(mosaico, TC_WET_L7,    'TC_Wetness')

    resultado = (ee.Image.cat([ndvi, tc_b, tc_g, tc_w])
                   .clip(region)
                   .cast({'NDVI': 'double', 'TC_Brightness': 'double',
                          'TC_Greenness': 'double', 'TC_Wetness': 'double'}))
    return resultado


### Exportar resultados a Google Drive

In [6]:
img_seca_l5, img_seca_l7, mosaico_seca_l5,   mosaico_seca_l7   = procesar_temporada('Seca',   *TEMPORADA_SECA)
img_humeda_l5, img_humeda_l7, mosaico_humeda_l5, mosaico_humeda_l7 = procesar_temporada('Húmeda', *TEMPORADA_HUMEDA)

# Exportar a Google Drive
def exportar(imagen, sensor, sufijo, escala):
    if imagen is None:
        print(f"Exportación cancelada para '{sensor}_{sufijo}': imagen no disponible.")
        return
    nombre_tarea = f'{PREFIJO}_{AÑO}_{sensor}_{sufijo}'
    tarea = ee.batch.Export.image.toDrive(
        image          = imagen,
        description    = nombre_tarea,
        folder         ='QLA_tif-2000',
        fileNamePrefix = nombre_tarea,
        region         = region,
        scale          = escala,
        crs            = f'EPSG:{CRS_UTM}',
        maxPixels      = 1e13,
        fileFormat     = 'GeoTIFF'
    )
    tarea.start()
    print(f"✓ Tarea iniciada: {nombre_tarea}")

exportar(img_seca_l5,   'L5', 'seca', ESCALA)
exportar(img_seca_l7,   'L7', 'seca', ESCALA)
exportar(img_humeda_l5, 'L5', 'humeda', ESCALA)
exportar(img_humeda_l7, 'L7', 'humeda', ESCALA)

# Exportar bandas corregidas (SR_B1_corr … SR_B7_corr)
exportar(mosaico_seca_l5,   'L5', 'seca_bandas_corr', ESCALA)
exportar(mosaico_seca_l7,   'L7', 'seca_bandas_corr', ESCALA)
exportar(mosaico_humeda_l5, 'L5', 'humeda_bandas_corr', ESCALA)
exportar(mosaico_humeda_l7, 'L7', 'humeda_bandas_corr', ESCALA)


── Temporada: Seca (03-01 → 05-31) ──
  Imágenes Landsat 5: 8
    · L5 2000-04-03
    · L5 2000-04-10
    · L5 2000-04-19
    · L5 2000-04-26
    · L5 2000-05-05
    · L5 2000-05-12
    · L5 2000-05-21
    · L5 2000-05-28
LT05_L2SP_029046_20000403_20200907_02_T1
LT05_L2SP_029046_20000419_20200907_02_T1
LT05_L2SP_029046_20000505_20200907_02_T1
LT05_L2SP_029046_20000521_20200907_02_T1
LT05_L2SP_030046_20000410_20200907_02_T1
LT05_L2SP_030046_20000426_20200907_02_T1
LT05_L2SP_030046_20000512_20200907_02_T1
LT05_L2SP_030046_20000528_20200907_02_T1
  Imágenes Landsat 7: 4
    · L7 2000-03-17
    · L7 2000-04-11
    · L7 2000-04-18
    · L7 2000-05-20
LE07_L2SP_029046_20000411_20200918_02_T1
LE07_L2SP_030046_20000317_20200918_02_T1
LE07_L2SP_030046_20000418_20200918_02_T1
LE07_L2SP_030046_20000520_20200918_02_T1
  L5 procesado correctamente
  L7 procesado correctamente

── Temporada: Húmeda (08-01 → 11-30) ──
  Imágenes Landsat 5: 11
    · L5 2000-08-09
    · L5 2000-08-16
    · L5 2000-08-

#Calcular estadisticas de capas y de indices

In [7]:
def calcular_estadisticas_indices(imagen, sensor, temporada):
    if imagen is None:
        print(f"  ℹ️ No hay datos para {sensor} - {temporada}")
        return
    nombre_base = f'{AÑO} {temporada} {sensor}'
    # Calcular y mostrar estadísticas
    print(f"\n{'='*60}")
    print(f"📊 Estadísticas para {nombre_base}")
    print(f"{'='*60}")

    for banda in ['NDVI', 'TC_Brightness', 'TC_Greenness', 'TC_Wetness']:
        stats = imagen.select(banda).reduceRegion(
            reducer=ee.Reducer.minMax(),
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).getInfo()

        # Combine the mean and standard deviation reducers.
        reducer_comb = ee.Reducer.mean().combine(
              reducer2=ee.Reducer.stdDev(), sharedInputs=True
        )
        stats2 = imagen.select(banda).reduceRegion(
            reducer=reducer_comb,
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).getInfo()

        min_val = stats.get(f'{banda}_min', 0)
        max_val = stats.get(f'{banda}_max', 0)
        mean_val = stats2.get(f'{banda}_mean', 0)
        std_val = stats2.get(f'{banda}_stdDev', 0)

        print(f"  {banda:>15}: min = {min_val:>8.4f} | max = {max_val:>8.4f} | rango = {max_val - min_val:>8.4f} | promedio = {mean_val:>8.4f} | SD = {std_val:>8.4f}")

    print(f"{'='*60}\n")


def calcular_estadisticas_bandas(imagen, sensor, temporada, bandas):
    if imagen is None:
        print(f"  ℹ️ No hay datos para {sensor} - {temporada}")
        return
    nombre_base = f'{AÑO} {temporada} {sensor}'
    # Calcular y mostrar estadísticas
    print(f"\n{'='*60}")
    print(f"📊 Estadísticas para {nombre_base}")
    print(f"{'='*60}")

    for banda in bandas:
        stats = imagen.select(banda).reduceRegion(
            reducer=ee.Reducer.minMax(),
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).getInfo()

        # Combine the mean and standard deviation reducers.
        reducer_comb = ee.Reducer.mean().combine(
              reducer2=ee.Reducer.stdDev(), sharedInputs=True
        )
        stats2 = imagen.select(banda).reduceRegion(
            reducer=reducer_comb,
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).getInfo()

        min_val = stats.get(f'{banda}_min', 0)
        max_val = stats.get(f'{banda}_max', 0)
        mean_val = stats2.get(f'{banda}_mean', 0)
        std_val = stats2.get(f'{banda}_stdDev', 0)

        print(f"  {banda:>15}: min = {min_val:>8.4f} | max = {max_val:>8.4f} | rango = {max_val - min_val:>8.4f} | promedio = {mean_val:>8.4f} | SD = {std_val:>8.4f}")

    print(f"{'='*60}\n")


print("Agregando capas al mapa (2000):")
calcular_estadisticas_indices(img_seca_l5, 'L5', 'Seca')
calcular_estadisticas_bandas(mosaico_seca_l5, 'L5', 'Seca', BANDAS_CORR)

calcular_estadisticas_indices(img_seca_l7, 'L7', 'Seca')
calcular_estadisticas_bandas(mosaico_seca_l7, 'L7', 'Seca', BANDAS_CORR)

calcular_estadisticas_indices(img_humeda_l5, 'L5', 'Húmeda')
calcular_estadisticas_bandas(mosaico_humeda_l5, 'L5', 'Húmeda', BANDAS_CORR)

calcular_estadisticas_indices(img_humeda_l7, 'L7', 'Húmeda')
calcular_estadisticas_bandas(mosaico_humeda_l7, 'L7', 'Húmeda', BANDAS_CORR)

Agregando capas al mapa (2000):

📊 Estadísticas para 2000 Seca L5
             NDVI: min =  -0.2023 | max =   0.7884 | rango =   0.9907 | promedio =   0.2476 | SD =   0.0859
    TC_Brightness: min =   0.0016 | max =   2.5807 | rango =   2.5791 | promedio =   0.5650 | SD =   0.1098
     TC_Greenness: min =  -0.4822 | max =   0.3400 | rango =   0.8222 | promedio =   0.0359 | SD =   0.0307
       TC_Wetness: min =  -1.3892 | max =   0.1878 | rango =   1.5770 | promedio =  -0.1898 | SD =   0.0539


📊 Estadísticas para 2000 Seca L5
       SR_B1_corr: min =   0.0401 | max =   1.9077 | rango =   1.8677 | promedio =   0.1060 | SD =   0.0237
       SR_B2_corr: min =   0.0589 | max =   0.6103 | rango =   0.5515 | promedio =   0.1411 | SD =   0.0343
       SR_B3_corr: min =   0.0537 | max =   0.6846 | rango =   0.6310 | promedio =   0.1799 | SD =   0.0479
       SR_B4_corr: min =   0.0477 | max =   0.9172 | rango =   0.8696 | promedio =   0.2944 | SD =   0.0538
       SR_B5_corr: min =   0.0238 |

### Visualización de imágenes procesadas

In [ ]:
# Recalcular centroide
gdf_bbox = gpd.read_file(RUTA_BBOX).to_crs(epsg=CRS_GEO)
gdf_area = gpd.read_file(RUTA_GEOJSON).to_crs(epsg=CRS_GEO)

# Calcular centroide en WGS84
# FIX: calcular centroide en CRS proyectado, luego volver a WGS84
centroid_proj = gdf_bbox.to_crs(epsg=3857).dissolve().centroid.iloc[0]
centroid = gpd.GeoSeries([centroid_proj], crs=3857).to_crs(epsg=4326).iloc[0]

#  MAPA
mapa = geemap.Map(height="950px", width="100%",layer_ctrl=False)
mapa.add_gdf(gdf_area, layer_name='ANP – Polígono',    style=STYLE_AREA)
mapa.centerObject(region, zoom=ZOOM)
mapa.add_gdf(gdf_bbox, layer_name='ANP – Bounding Box', style=STYLE_BBOX)
mapa.add_basemap('ROADMAP')

# PARÁMETROS DE VISUALIZACIÓN

# Landsat 5 TM (2010)
#No hay imagenes landsat 5 tm para el año 2010, pero hay para 2011
# Temporada SECA (basado en valores calculados)
VIS_L5_SECA = {
    'NDVI': {'min': -0.01, 'max': 0.83, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 0.80, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.13, 'max': 0.23, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.33, 'max': 0.30, 'palette': ['brown', 'white', 'blue']}
}

# Temporada HÚMEDA (basado en valores calculados)
VIS_L5_HUMEDA = {
    'NDVI': {'min': -0.13, 'max': 1.01, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 2.72, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.45, 'max': 0.66, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.26, 'max': 0.19, 'palette': ['brown', 'white', 'blue']}
}

# Landsat 7 ETM+ (2010)
VIS_L7_SECA = {
    'NDVI': {'min': -0.66, 'max': 1.02, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 1.12, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.68, 'max': 0.27, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.47, 'max': 0.47, 'palette': ['brown', 'white', 'blue']}
}

VIS_L7_HUMEDA = {
    'NDVI': {'min': -0.26, 'max': 0.93, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 0.94, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.20, 'max': 0.50, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.44, 'max': 0.11, 'palette': ['brown', 'white', 'blue']}
}


VISUALIZACION_2000 = {
    'L7': {
        'Seca': VIS_L7_SECA,
        'Húmeda': VIS_L7_HUMEDA
    },
    'L5': {
        'Seca': VIS_L5_SECA,
        'Húmeda': VIS_L5_HUMEDA
    }
}

# AGREGAR CAPAS CON PARÁMETROS DIFERENCIADOS
def agregar_capas(imagen, sensor, temporada):
    """Agrega capas al mapa con parámetros de visualización específicos por sensor y temporada"""
    if imagen is None:
        print(f"  ℹ️ No hay datos para {sensor} - {temporada}")
        return

    vis = VISUALIZACION_2000[sensor][temporada]
    nombre_base = f'2000 {temporada} {sensor}'

    # Definir qué capa mostrar inicialmente (ejemplo: L7 Seca)
    mostrar_inicial = (temporada == 'Seca' and sensor == 'L7')

    # Agregar capas
    for banda in ['NDVI', 'TC_Brightness', 'TC_Greenness', 'TC_Wetness']:
        mapa.addLayer(
            imagen.select(banda),
            vis[banda],
            f'{nombre_base} – {banda}',
            shown=(mostrar_inicial and banda == 'NDVI')
        )


print("Agregando capas al mapa:")
agregar_capas(img_seca_l5, 'L5', 'Seca')
agregar_capas(img_seca_l7, 'L7', 'Seca')
agregar_capas(img_humeda_l5, 'L5', 'Húmeda')
agregar_capas(img_humeda_l7, 'L7', 'Húmeda')

mapa.addLayerControl()
print("Visualización de imágenes procesadas")
mapa